# Superstore Sales SQL Analysis

Load the CSV into SQLite and run the core SQL checks for filtering, grouping, ranking, and data quality.

In [11]:
import sqlite3
from pathlib import Path

import pandas as pd
from IPython.display import display

base = Path(r'c:\Users\Subham Singhania\Desktop\celebal\Assignment-2')
csv_path = base / 'Sample - Superstore.csv'
db_path = base / 'superstore_sales.db'

df = pd.read_csv(csv_path, encoding='latin1')
df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
df['order_date'] = pd.to_datetime(df['order_date'], format='%m/%d/%Y').dt.strftime('%Y-%m-%d')
df['ship_date'] = pd.to_datetime(df['ship_date'], format='%m/%d/%Y').dt.strftime('%Y-%m-%d')

con = sqlite3.connect(db_path)
df.to_sql('sales', con, if_exists='replace', index=False)

for stmt in [
    'CREATE INDEX IF NOT EXISTS idx_sales_order_date ON sales(order_date)',
    'CREATE INDEX IF NOT EXISTS idx_sales_region ON sales(region)',
    'CREATE INDEX IF NOT EXISTS idx_sales_category ON sales(category)',
    'CREATE INDEX IF NOT EXISTS idx_sales_customer ON sales(customer_id)'
]:
    con.execute(stmt)
con.commit()

def q(sql):
    return pd.read_sql_query(sql, con)

In [12]:
schema = q('PRAGMA table_info(sales)')
sample = q('SELECT * FROM sales LIMIT 5')
quality = q('SELECT COUNT(*) AS rows, COUNT(DISTINCT row_id) AS distinct_rows, COUNT(DISTINCT order_id) AS distinct_orders, COUNT(DISTINCT customer_id) AS distinct_customers FROM sales')

display(schema)
display(sample)
display(quality)

,cid,name,type,notnull,dflt_value,pk
0,0,row_id,INTEGER,0,None,0
1,1,order_id,TEXT,0,None,0
2,2,order_date,TEXT,0,None,0
3,3,ship_date,TEXT,0,None,0
4,4,ship_mode,TEXT,0,None,0
5,5,customer_id,TEXT,0,None,0
6,6,customer_name,TEXT,0,None,0
7,7,segment,TEXT,0,None,0
8,8,country,TEXT,0,None,0
9,9,city,TEXT,0,None,0


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub-category,product_name,sales,quantity,discount,profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


,rows,distinct_rows,distinct_orders,distinct_customers
0,9994,9994,5009,793


In [13]:
west_technology = q("""
SELECT order_id, order_date, customer_name, region, category, product_name, sales, profit
FROM sales
WHERE region = 'West' AND category = 'Technology'
ORDER BY order_date
LIMIT 10
""")

high_sales = q("""
SELECT order_id, order_date, region, category, product_name, sales
FROM sales
WHERE sales > 1000
ORDER BY sales DESC
LIMIT 10
""")

year_2016_non_office = q("""
SELECT order_id, order_date, region, category, sales
FROM sales
WHERE order_date BETWEEN '2016-01-01' AND '2016-12-31'
  AND category <> 'Office Supplies'
ORDER BY order_date
""")

display(west_technology)
display(high_sales)
display(year_2016_non_office)

,order_id,order_date,customer_name,region,category,product_name,sales,profit
0,CA-2014-121762,2014-02-14,Marina Lichtenstein,West,Technology,Logitech G600 MMO Gaming Mouse,239.970,86.3892
1,CA-2014-114510,2014-03-14,Jason Fortune-,West,Technology,Imation 30456 USB Flash Drive 8GB,82.800,6.6240
2,US-2014-131275,2014-03-18,Sample Company A,West,Technology,Swingline SM12-08 MicroCut Jam Free Shredder,1279.968,415.9896
3,CA-2014-120838,2014-03-23,Patrick O'Donnell,West,Technology,Ooma Telo VoIP Home Phone System,604.752,37.7970
4,CA-2014-128237,2014-03-25,Christina Anderson,West,Technology,Imation Swivel Flash Drive USB flash drive - 8 GB,45.480,15.9180
5,CA-2014-138436,2014-03-26,Jonathan Doherty,West,Technology,SanDisk Ultra 32 GB MicroSDHC Class 10 Memory ...,66.300,8.6190
6,CA-2014-141838,2014-03-26,Damala Kotsonis,West,Technology,Griffin GC17055 Auxiliary Audio Cable,28.784,2.8784
7,CA-2014-100881,2014-03-28,Daniel Raglin,West,Technology,AT&T TR1909W,302.376,22.6782
8,CA-2014-112291,2014-04-03,Katrina Edelman,West,Technology,Enermax Briskie RF Wireless Keyboard and Mouse...,62.310,22.4316
9,CA-2014-112291,2014-04-03,Katrina Edelman,West,Technology,Logitech G600 MMO Gaming Mouse,159.980,57.5928


,order_id,order_date,region,category,product_name,sales
0,CA-2014-145317,2014-03-18,South,Technology,Cisco TelePresence System EX90 Videoconferenci...,22638.480
1,CA-2016-118689,2016-10-02,Central,Technology,Canon imageCLASS 2200 Advanced Copier,17499.950
2,CA-2017-140151,2017-03-23,West,Technology,Canon imageCLASS 2200 Advanced Copier,13999.960
3,CA-2017-127180,2017-10-22,East,Technology,Canon imageCLASS 2200 Advanced Copier,11199.968
4,CA-2017-166709,2017-11-17,East,Technology,Canon imageCLASS 2200 Advanced Copier,10499.970
5,CA-2016-117121,2016-12-17,Central,Office Supplies,GBC Ibimaster 500 Manual ProClick Binding System,9892.740
6,CA-2014-116904,2014-09-23,Central,Office Supplies,Ibico EPK-21 Electric Binding System,9449.950
7,US-2016-107440,2016-04-16,East,Technology,"3D Systems Cube Printer, 2nd Generation, Magenta",9099.930
8,CA-2016-158841,2016-02-02,South,Technology,HP Designjet T520 Inkjet Large Format Printer ...,8749.950
9,CA-2016-143714,2016-05-23,East,Technology,Canon imageCLASS 2200 Advanced Copier,8399.976


,order_id,order_date,region,category,sales
0,CA-2016-160304,2016-01-02,East,Furniture,173.940
1,CA-2016-160304,2016-01-02,East,Technology,231.980
2,US-2016-116365,2016-01-03,Central,Technology,30.080
3,US-2016-116365,2016-01-03,Central,Technology,165.600
4,US-2016-116365,2016-01-03,Central,Technology,180.960
...,...,...,...,...,...
1016,CA-2016-138667,2016-12-27,East,Technology,40.000
1017,CA-2016-156300,2016-12-29,Central,Furniture,754.450
1018,CA-2016-122017,2016-12-29,Central,Furniture,70.560
1019,CA-2016-105746,2016-12-30,East,Furniture,170.786


In [14]:
by_category = q("""
SELECT category,
       COUNT(*) AS line_items,
       ROUND(SUM(sales), 2) AS total_sales,
       SUM(quantity) AS total_quantity,
       ROUND(AVG(sales), 2) AS avg_sales,
       ROUND(AVG(profit), 2) AS avg_profit
FROM sales
GROUP BY category
ORDER BY total_sales DESC
""")

by_region = q("""
SELECT region,
       COUNT(*) AS line_items,
       ROUND(SUM(sales), 2) AS total_sales,
       SUM(quantity) AS total_quantity,
       ROUND(AVG(sales), 2) AS avg_sales
FROM sales
GROUP BY region
ORDER BY total_sales DESC
""")

by_year = q("""
SELECT substr(order_date, 1, 4) AS order_year,
       COUNT(*) AS line_items,
       ROUND(SUM(sales), 2) AS total_sales,
       ROUND(AVG(sales), 2) AS avg_sales
FROM sales
GROUP BY substr(order_date, 1, 4)
ORDER BY order_year
""")

display(by_category)
display(by_region)
display(by_year)

,category,line_items,total_sales,total_quantity,avg_sales,avg_profit
0,Technology,1847,836154.03,6939,452.71,78.75
1,Furniture,2121,741999.80,8028,349.83,8.70
2,Office Supplies,6026,719047.03,22906,119.32,20.33


,region,line_items,total_sales,total_quantity,avg_sales
0,West,3203,725457.82,12266,226.49
1,East,2848,678781.24,10618,238.34
2,Central,2323,501239.89,8780,215.77
3,South,1620,391721.91,6209,241.80


,order_year,line_items,total_sales,avg_sales
0,2014,1993,484247.50,242.97
1,2015,2102,470532.51,223.85
2,2016,2587,609205.60,235.49
3,2017,3312,733215.26,221.38


In [15]:
top_products = q("""
SELECT product_name,
       ROUND(SUM(sales), 2) AS total_sales,
       SUM(quantity) AS total_quantity
FROM sales
GROUP BY product_name
ORDER BY total_sales DESC, total_quantity DESC
LIMIT 10
""")

top_categories = q("""
SELECT category,
       ROUND(SUM(sales), 2) AS total_sales
FROM sales
GROUP BY category
ORDER BY total_sales DESC
LIMIT 5
""")

top_customers = q("""
SELECT customer_id,
       customer_name,
       COUNT(DISTINCT order_id) AS orders,
       ROUND(SUM(sales), 2) AS total_sales,
       ROUND(AVG(sales), 2) AS avg_line_sales
FROM sales
GROUP BY customer_id, customer_name
ORDER BY total_sales DESC
LIMIT 10
""")

display(top_products)
display(top_categories)
display(top_customers)

,product_name,total_sales,total_quantity
0,Canon imageCLASS 2200 Advanced Copier,61599.82,20
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.38,31
2,Cisco TelePresence System EX90 Videoconferenci...,22638.48,6
3,HON 5400 Series Task Chairs for Big and Tall,21870.58,39
4,GBC DocuBind TL300 Electric Binding System,19823.48,37
5,GBC Ibimaster 500 Manual ProClick Binding System,19024.50,48
6,Hewlett Packard LaserJet 3310 Copier,18839.69,38
7,HP Designjet T520 Inkjet Large Format Printer ...,18374.90,12
8,GBC DocuBind P400 Electric Binding System,17965.07,27
9,High Speed Automatic Electric Letter Opener,17030.31,11


,category,total_sales
0,Technology,836154.03
1,Furniture,741999.80
2,Office Supplies,719047.03


,customer_id,customer_name,orders,total_sales,avg_line_sales
0,SM-20320,Sean Miller,5,25043.05,1669.54
1,TC-20980,Tamara Chand,5,19052.22,1587.68
2,RB-19360,Raymond Buch,6,15117.34,839.85
3,TA-21385,Tom Ashbrook,4,14595.62,1459.56
4,AB-10105,Adrian Barton,10,14473.57,723.68
5,KL-16645,Ken Lonsdale,12,14175.23,488.80
6,SC-20095,Sanjit Chand,9,14142.33,642.83
7,HL-15040,Hunter Lopez,6,12873.30,1170.30
8,SE-20110,Sanjit Engle,11,12209.44,642.60
9,CC-12370,Christopher Conant,5,12129.07,1102.64


In [16]:
monthly_trend = q("""
SELECT substr(order_date, 1, 7) AS month,
       COUNT(DISTINCT order_id) AS orders,
       COUNT(*) AS line_items,
       ROUND(SUM(sales), 2) AS total_sales,
       ROUND(SUM(profit), 2) AS total_profit
FROM sales
GROUP BY substr(order_date, 1, 7)
ORDER BY month
""")

dup_rows = q("""
SELECT order_id, product_id, COUNT(*) AS rows_found
FROM sales
GROUP BY order_id, product_id
HAVING COUNT(*) > 1
ORDER BY rows_found DESC
""")

null_check = q("""
SELECT
  SUM(CASE WHEN row_id IS NULL THEN 1 ELSE 0 END) AS row_id_nulls,
  SUM(CASE WHEN order_id IS NULL THEN 1 ELSE 0 END) AS order_id_nulls,
  SUM(CASE WHEN customer_id IS NULL THEN 1 ELSE 0 END) AS customer_id_nulls,
  SUM(CASE WHEN sales IS NULL THEN 1 ELSE 0 END) AS sales_nulls,
  SUM(CASE WHEN profit IS NULL THEN 1 ELSE 0 END) AS profit_nulls
FROM sales
""")

display(monthly_trend)
display(dup_rows)
display(null_check)

,month,orders,line_items,total_sales,total_profit
0,2014-01,32,79,14236.90,2450.19
1,2014-02,28,46,4519.89,862.31
2,2014-03,71,157,55691.01,498.73
3,2014-04,66,135,28295.35,3488.84
4,2014-05,69,122,23648.29,2738.71
5,2014-06,66,135,34595.13,4976.52
6,2014-07,65,143,33946.39,-841.48
7,2014-08,72,153,27909.47,5318.10
8,2014-09,130,268,81777.35,8328.10
9,2014-10,78,159,31453.39,3448.26


,order_id,product_id,rows_found
0,CA-2015-103135,OFF-BI-10000069,2
1,CA-2016-129714,OFF-PA-10001970,2
2,CA-2016-137043,FUR-FU-10003664,2
3,CA-2016-140571,OFF-PA-10001954,2
4,CA-2017-118017,TEC-AC-10002006,2
5,CA-2017-152912,OFF-ST-10003208,2
6,US-2014-150119,FUR-CH-10002965,2
7,US-2016-123750,TEC-AC-10004659,2


,row_id_nulls,order_id_nulls,customer_id_nulls,sales_nulls,profit_nulls
0,0,0,0,0,0


## Quick takeaways

- Technology leads revenue, with Furniture and Office Supplies close behind.
- The West region brings the highest sales in this sample, and 2017 is the strongest year.
- Monthly sales become easy to track once the date field is stored in ISO format.
- Row ID is unique, but the duplicate check shows repeated order/product pairs in the raw data.